#### 02 — SQL Analysis
**Goal:** Run all 7 SQL files from Python, display results as tables,
and export CSVs for Power BI.

This notebook runs every query and captures the output.
Each section maps to one `.sql` file.

In [1]:
!pip install pandas
import pandas as pd
import sqlite3
import os

# Connect to the db built in notebook 1:
conn = sqlite3.connect(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\data\retail_rocket.db')
os.makedirs(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs', exist_ok = True)

def run_sql(filepath):
    """Read a .sql file and return a list of DataFrames,
    one per SELECT statement inside the file."""
    with open(filepath, 'r') as f:
        raw = f.head()
    # Split on semicolons, keep non-empty SELECT statements
    statements = [s.strip() for s in raw.split(';')
                    if s.strip().upper().startswith('SELECT')]
    results = []

    for stmt in statement:
        try:
            df = pd.read_sql(stmt, conn)
            results.append(df)
        except Exception as e:
            print(f"  Skipped (not a SELECT or error): {e}")
    return results

print("Connected to retail_rocket.db")


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Connected to retail_rocket.db


##### Section 1 — Raw Data Exploration (01_setup.sql)


In [2]:
print('====== Total Events ======')
df = pd.read_sql('select count(*) as total_events from events', conn)
display(df)

print('\n=== EVENT TYPE BREAKDOWN ===')
df = pd.read_sql("""
            select event_type,
                 count(*) as event_count,
                 round(100.0 * count(*) / sum(count(*)) over(), 2) as pct_total
            from events
            group by event_type
            order by event_count desc""", conn)
display(df)

print("\n=== DATE RANGE ===")

df = pd.read_sql("""
            select
                min(event_time) as earliest,
                max(event_time) as latest,
                cast(julianday(max(event_time)) - julianday(min(event_time)) as int) as total_days
            from events""", conn)
display(df)

print("\n=== UNIQUE USERS AND ITEMS ===")
df = pd.read_sql("""
            select
                count(distinct user_id) as unique_users,
                count(distinct item_id) as unique_items
            from events""", conn)
display(df)

====== Total Events ======


,total_events
0,2755641



=== EVENT TYPE BREAKDOWN ===


,event_type,event_count,pct_total
0,view,2664218,96.68
1,addtocart,68966,2.50
2,transaction,22457,0.81



=== DATE RANGE ===


,earliest,latest,total_days
0,2015-05-03 03:00:04,2015-09-18 02:59:47,137



=== UNIQUE USERS AND ITEMS ===


,unique_users,unique_items
0,1407580,235061


##### Section 2 — Cohort Assignment (02_cohort_assignment.sql)

In [3]:
# Drop and recreate if it already exists
conn.execute("drop table if exists user_cohorts")

conn.execute("""
create table user_cohorts as 
with first_seen as (
    select
        user_id,
        date(min(event_time)) as first_date,
        strftime('%Y-%m', min(event_time)) as cohort_month
    from events
    group by user_id
)
select  
    user_id,
    first_date,
    cohort_month
from first_seen
""")

conn.commit()

# Preview cohort size
cohort_sizes =  pd.read_sql("""
    select
        cohort_month,
        count(distinct user_id) as cohort_size
    from user_cohorts
    group by cohort_month
    order by cohort_month""", conn)

display(cohort_sizes)
cohort_sizes.to_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs\cohort_sizes.csv')

,cohort_month,cohort_size
0,2015-05,307574
1,2015-06,299189
2,2015-07,355765
3,2015-08,288328
4,2015-09,156724


##### Section 3 — Retention Matrix (03_retention_matrix.sql)
**Business Question 1:** What % of users return in months 1, 2, 3, 4? <br>
**Business Question 2:** Which cohort has the best month-2 retention?

In [4]:
retention_raw = pd.read_sql("""
                    with monthly_activity as (
                        select
                            e.user_id,
                            c.cohort_month,
                            strftime('%Y-%m', e.event_time) as activity_month
                        from events e join user_cohorts c on e.user_id = c.user_id
                    ),
                    month_offsets AS (
                        SELECT user_id, cohort_month,
                            (CAST(strftime('%Y', activity_month||'-01') AS INT)*12 +
                            CAST(strftime('%m', activity_month||'-01') AS INT)) -
                            (CAST(strftime('%Y', cohort_month  ||'-01') AS INT)*12 +
                            CAST(strftime('%m', cohort_month  ||'-01') AS INT)) AS month_number
                        FROM monthly_activity
                    ),
                    cohort_sizes AS (
                        SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_size
                        FROM user_cohorts GROUP BY cohort_month
                    ),
                    retention_counts AS (
                        SELECT cohort_month, month_number,
                            COUNT(DISTINCT user_id) AS retained_users
                        FROM month_offsets WHERE month_number BETWEEN 0 AND 4
                        GROUP BY cohort_month, month_number
                    )
                    SELECT r.cohort_month, cs.cohort_size,
                        r.month_number, r.retained_users,
                        ROUND(100.0 * r.retained_users / cs.cohort_size, 1) AS retention_pct
                    FROM retention_counts r
                    JOIN cohort_sizes cs ON r.cohort_month = cs.cohort_month
                    ORDER BY r.cohort_month, r.month_number
                    """, conn)

# Pivot into matrix format for Power BI and heatmap
retention_matrix = retention_raw.pivot(
    index = 'cohort_month', columns = 'month_number', values = 'retention_pct'
)

retention_matrix.columns = [f'month_{c}' for c in retention_matrix.columns]
display(retention_matrix)
retention_matrix.to_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs\retention_matrix.csv')
retention_raw.to_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs\retention_raw.csv', index = False)
print('Retention matrix exported')

,month_0,month_1,month_2,month_3,month_4
cohort_month,,,,,
2015-05,100.0,4.8,2.9,1.9,1.1
2015-06,100.0,4.2,2.0,1.0,NaN
2015-07,100.0,3.1,1.1,NaN,NaN
2015-08,100.0,2.4,NaN,NaN,NaN
2015-09,100.0,NaN,NaN,NaN,NaN


Retention matrix exported


In [5]:
print("=== BUSINESS QUESTION 2: Best month-2 retention by cohort ===")
best_cohort = retention_raw[retention_raw['month_number'] == 2].sort_values('retention_pct', ascending = False)
display(best_cohort[['cohort_month','cohort_size','retained_users','retention_pct']])

=== BUSINESS QUESTION 2: Best month-2 retention by cohort ===


,cohort_month,cohort_size,retained_users,retention_pct
2,2015-05,307574,8783,2.9
7,2015-06,299189,6047,2.0
11,2015-07,355765,3884,1.1


##### Section 4 — Funnel Analysis (04_funnel_analysis.sql)
**Business Question 3:** Overall funnel drop-off<br>
**Business Question 4:** Conversion by item category<br>
**Business Question 5:** Conversion by day of week<br>
**Business Question 6:** Conversion by hour of day<br>

In [6]:
print("=== BQ3: Overall Funnel ===")
funnel = pd.read_sql("""
    SELECT
        COUNT(DISTINCT CASE WHEN event_type='view'        THEN user_id END) AS viewers,
        COUNT(DISTINCT CASE WHEN event_type='addtocart'   THEN user_id END) AS carted,
        COUNT(DISTINCT CASE WHEN event_type='transaction' THEN user_id END) AS purchased,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN event_type='addtocart'   THEN user_id END) /
              COUNT(DISTINCT CASE WHEN event_type='view' THEN user_id END), 1) AS view_to_cart_pct,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN event_type='transaction' THEN user_id END) /
              NULLIF(COUNT(DISTINCT CASE WHEN event_type='addtocart' THEN user_id END),0),1) AS cart_to_buy_pct,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN event_type='transaction' THEN user_id END) /
              COUNT(DISTINCT CASE WHEN event_type='view' THEN user_id END), 1) AS overall_conv_pct
    FROM events
""", conn)
display(funnel)

# Reshape for Power BI funnel chart
funnel_long = pd.DataFrame({
    'stage' : ['Viewed','Added to Cart','Purchased'],
    'stage_order' : [1,2,3],
    'users' : [funnel['viewers'][0], funnel['carted'][0], funnel['purchased'][0]]
})

funnel_long['drop_off_pct'] = (
    (1 - funnel_long['users'] / funnel_long['users'].shift(1)) * 100
).round(1)

funnel_long.to_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs\funnel.csv', index = False)
display(funnel_long)

=== BQ3: Overall Funnel ===


,viewers,carted,purchased,view_to_cart_pct,cart_to_buy_pct,overall_conv_pct
0,1404179,37722,11719,2.7,31.1,0.8


,stage,stage_order,users,drop_off_pct
0,Viewed,1,1404179,NaN
1,Added to Cart,2,37722,97.3
2,Purchased,3,11719,68.9


In [7]:
print("=== BQ5: Conversion by Day of Week ===")
dow = pd.read_sql("""
    SELECT day_of_week,
           COUNT(DISTINCT CASE WHEN event_type='view'
                 THEN user_id END)                  AS viewers,
           COUNT(DISTINCT CASE WHEN event_type='transaction'
                 THEN user_id END)                  AS buyers,
           ROUND(100.0 *
               COUNT(DISTINCT CASE WHEN event_type='transaction'
                     THEN user_id END) /
               NULLIF(COUNT(DISTINCT CASE WHEN event_type='view'
                     THEN user_id END), 0), 2)      AS conversion_pct
    FROM events
    GROUP BY day_of_week
    ORDER BY conversion_pct DESC
""", conn)

# Add day number for correct sorting (Monday=0, Sunday=6)
day_order = {
    'Monday':    0,
    'Tuesday':   1,
    'Wednesday': 2,
    'Thursday':  3,
    'Friday':    4,
    'Saturday':  5,
    'Sunday':    6
}
dow['day_number'] = dow['day_of_week'].map(day_order)
dow = dow.sort_values('day_number')
dow.to_csv('../outputs/day_of_week_conversion.csv', index=False)
display(dow)
print("✅ day_of_week_conversion.csv exported with day_number column")

=== BQ5: Conversion by Day of Week ===


,day_of_week,viewers,buyers,conversion_pct,day_number
2,Monday,251617,2190,0.87,0
1,Tuesday,254023,2262,0.89,1
0,Wednesday,245705,2244,0.91,2
3,Thursday,239442,1978,0.83,3
4,Friday,220530,1600,0.73,4
6,Saturday,181604,1052,0.58,5
5,Sunday,197384,1241,0.63,6


✅ day_of_week_conversion.csv exported with day_number column


##### Section 5 — Churn Flags (05_churn_flags.sql)
**Business Question 7:** Churn segment distribution<br>
**Business Question 8:** Cart abandoner inactivity window

In [8]:
conn.execute("DROP TABLE IF EXISTS churn_labels")
conn.execute("""
CREATE TABLE churn_labels AS
WITH last_seen AS (
    SELECT user_id, MAX(DATE(event_time)) AS last_active_date,
           COUNT(*) AS total_events, COUNT(DISTINCT DATE(event_time)) AS active_days
    FROM events GROUP BY user_id
),
purchase_history AS (
    SELECT user_id,
           MAX(CASE WHEN event_type='transaction' THEN 1 ELSE 0 END) AS ever_purchased,
           COUNT(CASE WHEN event_type='transaction' THEN 1 END)       AS total_purchases,
           MAX(CASE WHEN event_type='addtocart'   THEN 1 ELSE 0 END) AS ever_carted
    FROM events GROUP BY user_id
)
SELECT l.user_id, l.last_active_date, l.total_events, l.active_days,
       p.ever_purchased, p.total_purchases, p.ever_carted,
       CAST(julianday('2015-09-18')-julianday(l.last_active_date) AS INT) AS days_since_active,
       CASE
           WHEN CAST(julianday('2015-09-18')-julianday(l.last_active_date) AS INT)<=30
            AND p.ever_purchased=1                      THEN 'active_buyer'
           WHEN CAST(julianday('2015-09-18')-julianday(l.last_active_date) AS INT)<=30
            AND p.ever_purchased=0                      THEN 'active_browser'
           WHEN CAST(julianday('2015-09-18')-julianday(l.last_active_date) AS INT)>30
            AND p.ever_purchased=1                      THEN 'churned_buyer'
           WHEN CAST(julianday('2015-09-18')-julianday(l.last_active_date) AS INT)>30
            AND p.ever_carted=1 AND p.ever_purchased=0 THEN 'churned_cart_abandoner'
           ELSE                                              'churned_viewer'
       END AS churn_label
FROM last_seen l JOIN purchase_history p ON l.user_id=p.user_id
""")
conn.commit()
print("✅ churn_labels table created")

print("\n=== BQ7: Churn Segment Distribution ===")
churn_segments = pd.read_sql("""
    SELECT churn_label, COUNT(*) AS user_count,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),1) AS pct_of_users
    FROM churn_labels GROUP BY churn_label ORDER BY user_count DESC
""", conn)
display(churn_segments)
churn_segments.to_csv('../outputs/churn_segments.csv', index=False)

print("\n=== BQ8: Cart Abandoner Inactivity Window ===")
cart_inactivity = pd.read_sql("""
    SELECT CASE
               WHEN days_since_active BETWEEN 30 AND 44 THEN '30-44 days'
               WHEN days_since_active BETWEEN 45 AND 59 THEN '45-59 days'
               WHEN days_since_active BETWEEN 60 AND 89 THEN '60-89 days'
               ELSE '90+ days'
           END AS inactivity_bucket,
           COUNT(*) AS user_count
    FROM churn_labels WHERE churn_label='churned_cart_abandoner'
    GROUP BY inactivity_bucket ORDER BY MIN(days_since_active)
""", conn)
display(cart_inactivity)
cart_inactivity.to_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs\cart_abandoner_inactivity.csv', index=False)

✅ churn_labels table created

=== BQ7: Churn Segment Distribution ===


,churn_label,user_count,pct_of_users
0,churned_viewer,1077790,76.6
1,active_browser,297269,21.1
2,churned_cart_abandoner,20802,1.5
3,churned_buyer,8802,0.6
4,active_buyer,2917,0.2



=== BQ8: Cart Abandoner Inactivity Window ===


,inactivity_bucket,user_count
0,30-44 days,2764
1,45-59 days,3356
2,60-89 days,5939
3,90+ days,8743


##### Section 6 — Power Users (06_power_users.sql)
**Business Question 9:** Future purchase rate by week-1 segment<br>
**Business Question 10:** Event threshold that separates high vs low retention

In [9]:
conn.execute("DROP TABLE IF EXISTS user_segments")
conn.execute("""
CREATE TABLE user_segments AS
WITH week1_activity AS (
    SELECT e.user_id,
           COUNT(*)                                                     AS w1_total_events,
           COUNT(DISTINCT DATE(e.event_time))                           AS w1_active_days,
           COUNT(DISTINCT e.item_id)                                    AS w1_unique_items,
           COUNT(CASE WHEN e.event_type='addtocart'   THEN 1 END)      AS w1_cart_actions,
           COUNT(CASE WHEN e.event_type='transaction' THEN 1 END)      AS w1_purchases,
           ROUND(1.0*COUNT(*)/NULLIF(COUNT(DISTINCT DATE(e.event_time)),0),1) AS w1_events_per_day
    FROM events e JOIN user_cohorts c ON e.user_id=c.user_id
    WHERE julianday(DATE(e.event_time))-julianday(c.first_date) BETWEEN 0 AND 7
    GROUP BY e.user_id
),
later_behaviour AS (
    SELECT e.user_id,
           MAX(CASE WHEN e.event_type='transaction'
                AND julianday(DATE(e.event_time))-julianday(c.first_date)>7
                THEN 1 ELSE 0 END) AS purchased_after_week1
    FROM events e JOIN user_cohorts c ON e.user_id=c.user_id
    GROUP BY e.user_id
)
SELECT w.user_id, w.w1_total_events, w.w1_active_days, w.w1_unique_items,
       w.w1_cart_actions, w.w1_purchases, w.w1_events_per_day,
       l.purchased_after_week1,
       CASE WHEN w.w1_purchases>=2   THEN 'power_buyer'
            WHEN w.w1_purchases=1    THEN 'single_buyer'
            WHEN w.w1_cart_actions>=1 THEN 'cart_browser'
            WHEN w.w1_unique_items>=10 THEN 'heavy_viewer'
            ELSE                         'light_viewer' END AS user_segment
FROM week1_activity w JOIN later_behaviour l ON w.user_id=l.user_id
""")
conn.commit()
print("✅ user_segments table created")

print("\n=== BQ9: Future Purchase Rate by Segment ===")
segments = pd.read_sql("""
    SELECT user_segment, COUNT(*) AS total_users,
           SUM(purchased_after_week1) AS later_buyers,
           ROUND(100.0*SUM(purchased_after_week1)/COUNT(*),1) AS future_purchase_rate_pct,
           ROUND(AVG(w1_total_events),1) AS avg_w1_events,
           ROUND(AVG(w1_active_days),1)  AS avg_active_days
    FROM user_segments GROUP BY user_segment ORDER BY future_purchase_rate_pct DESC
""", conn)
display(segments)
segments.to_csv('../outputs/user_segments.csv', index=False)

print("\n=== BQ10: Event Threshold Analysis ===")
threshold = pd.read_sql("""
    SELECT CASE WHEN w1_total_events=1              THEN '01 event'
                WHEN w1_total_events BETWEEN 2 AND 4  THEN '02-04 events'
                WHEN w1_total_events BETWEEN 5 AND 9  THEN '05-09 events'
                WHEN w1_total_events BETWEEN 10 AND 19 THEN '10-19 events'
                ELSE '20+ events' END AS event_bucket,
           COUNT(*) AS users,
           ROUND(100.0*SUM(purchased_after_week1)/COUNT(*),1) AS future_purchase_rate_pct
    FROM user_segments GROUP BY event_bucket ORDER BY MIN(w1_total_events)
""", conn)
display(threshold)
threshold.to_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Retail Rocket - User Retention & Churn Analysis\outputs\event_threshold.csv', index=False)

✅ user_segments table created

=== BQ9: Future Purchase Rate by Segment ===


,user_segment,total_users,later_buyers,future_purchase_rate_pct,avg_w1_events,avg_active_days
0,power_buyer,2099,113,5.4,32.8,1.8
1,single_buyer,8466,137,1.6,6.7,1.4
2,heavy_viewer,4271,56,1.3,21.7,1.9
3,cart_browser,25668,160,0.6,5.7,1.3
4,light_viewer,1367076,938,0.1,1.5,1.1



=== BQ10: Event Threshold Analysis ===


,event_bucket,users,future_purchase_rate_pct
0,01 event,1047086,0.0
1,02-04 events,295874,0.1
2,05-09 events,47826,0.4
3,10-19 events,12723,1.2
4,20+ events,4071,4.2


##### Section 7 — Advanced Business Questions (07_business_questions.sql)
**BQ11:** Time-to-conversion (how many events before first purchase)<br>
**BQ12:** Cart-to-purchase decision speed<br>
**BQ13:** Repeat purchase rate<br>
**BQ14:** Month-over-month growth<br>
**BQ15:** 7-day rolling transaction trend<br>

In [10]:
print("=== BQ11: Events Before First Purchase ===")
ttp = pd.read_sql("""
WITH events_ordered AS (
    SELECT user_id, event_type,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS event_rank
    FROM events
),
first_purchase AS (
    SELECT user_id, MIN(event_rank) AS events_before_purchase
    FROM events_ordered WHERE event_type='transaction' GROUP BY user_id
)
SELECT CASE WHEN events_before_purchase=1               THEN '1st event'
            WHEN events_before_purchase BETWEEN 2 AND 5   THEN '2-5 events'
            WHEN events_before_purchase BETWEEN 6 AND 20  THEN '6-20 events'
            WHEN events_before_purchase BETWEEN 21 AND 50 THEN '21-50 events'
            ELSE '50+ events' END AS touchpoints,
       COUNT(*) AS buyers,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),1) AS pct_of_buyers
FROM first_purchase GROUP BY touchpoints ORDER BY MIN(events_before_purchase)
""", conn)
display(ttp)
ttp.to_csv('../outputs/time_to_conversion.csv', index=False)

print("\n=== BQ12: Cart-to-Purchase Speed ===")
speed = pd.read_sql("""
WITH cart_to_buy AS (
    SELECT user_id, item_id, event_time AS cart_time,
           LEAD(event_time)  OVER (PARTITION BY user_id,item_id ORDER BY event_time) AS next_time,
           LEAD(event_type)  OVER (PARTITION BY user_id,item_id ORDER BY event_time) AS next_event
    FROM events WHERE event_type='addtocart'
)
SELECT CASE WHEN (julianday(next_time)-julianday(cart_time))*24<1  THEN 'Under 1 hour'
            WHEN (julianday(next_time)-julianday(cart_time))*24<24 THEN '1-24 hours'
            WHEN (julianday(next_time)-julianday(cart_time))*24<72 THEN '1-3 days'
            ELSE 'Over 3 days' END AS time_to_purchase,
       COUNT(*) AS conversions
FROM cart_to_buy WHERE next_event='transaction'
GROUP BY time_to_purchase ORDER BY MIN((julianday(next_time)-julianday(cart_time))*24)
""", conn)
display(speed)
speed.to_csv('../outputs/cart_to_purchase_speed.csv', index=False)

print("\n=== BQ13: Repeat Purchase Rate ===")
repeat = pd.read_sql("""
WITH pc AS (SELECT user_id, COUNT(DISTINCT transaction_id) AS num_purchases
            FROM events WHERE event_type='transaction' AND transaction_id IS NOT NULL
            GROUP BY user_id)
SELECT CASE WHEN num_purchases=1  THEN 'One-time buyer'
            WHEN num_purchases=2  THEN 'Bought twice'
            ELSE 'Loyal (3+)' END AS buyer_type,
       COUNT(*) AS users,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),1) AS pct_of_buyers
FROM pc GROUP BY buyer_type ORDER BY MIN(num_purchases)
""", conn)
display(repeat)
repeat.to_csv('../outputs/repeat_purchase_rate.csv', index=False)

print("\n=== BQ14: Month-over-Month Growth ===")
growth = pd.read_sql("""
WITH monthly AS (
    SELECT strftime('%Y-%m', event_time) AS month,
           COUNT(DISTINCT user_id) AS total_users,
           COUNT(CASE WHEN event_type='transaction' THEN 1 END) AS transactions
    FROM events GROUP BY month
)
SELECT month, total_users, transactions,
       LAG(total_users)   OVER (ORDER BY month) AS prev_users,
       LAG(transactions)  OVER (ORDER BY month) AS prev_transactions,
       ROUND(100.0*(total_users-LAG(total_users) OVER (ORDER BY month))/
             NULLIF(LAG(total_users) OVER (ORDER BY month),0),1) AS user_growth_pct,
       ROUND(100.0*(transactions-LAG(transactions) OVER (ORDER BY month))/
             NULLIF(LAG(transactions) OVER (ORDER BY month),0),1) AS txn_growth_pct
FROM monthly ORDER BY month
""", conn)
display(growth)
growth.to_csv('../outputs/monthly_growth.csv', index=False)

print("\n=== BQ15: 7-Day Rolling Transactions ===")
rolling = pd.read_sql("""
WITH daily AS (
    SELECT DATE(event_time) AS txn_date, COUNT(*) AS daily_transactions
    FROM events WHERE event_type='transaction' GROUP BY txn_date
)
SELECT txn_date, daily_transactions,
       ROUND(AVG(daily_transactions) OVER (
           ORDER BY txn_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),1) AS rolling_7d_avg
FROM daily ORDER BY txn_date
""", conn)
display(rolling)
rolling.to_csv('../outputs/rolling_transactions.csv', index=False)

conn.close()
print("\n✅ All 15 business questions answered. All CSVs exported to outputs/")

=== BQ11: Events Before First Purchase ===


,touchpoints,buyers,pct_of_buyers
0,1st event,118,1.0
1,2-5 events,7054,60.2
2,6-20 events,3821,32.6
3,21-50 events,572,4.9
4,50+ events,154,1.3



=== BQ12: Cart-to-Purchase Speed ===


,time_to_purchase,conversions



=== BQ13: Repeat Purchase Rate ===


...


=== BQ14: Month-over-Month Growth ===


,month,total_users,transactions,prev_users,prev_transactions,user_growth_pct,txn_growth_pct
0,2015-05,307574,4611,NaN,NaN,NaN,NaN
1,2015-06,313832,5043,307574.0,4611.0,2.0,9.4
2,2015-07,377199,5802,313832.0,5043.0,20.2,15.1
3,2015-08,311128,4632,377199.0,5802.0,-17.5,-20.2
4,2015-09,173728,2369,311128.0,4632.0,-44.2,-48.9



=== BQ15: 7-Day Rolling Transactions ===


,txn_date,daily_transactions,rolling_7d_avg
0,2015-05-03,83,83.0
1,2015-05-04,154,118.5
2,2015-05-05,225,154.0
3,2015-05-06,258,180.0
4,2015-05-07,217,187.4
...,...,...,...
134,2015-09-14,154,153.7
135,2015-09-15,158,155.1
136,2015-09-16,137,147.3
137,2015-09-17,45,127.9



✅ All 15 business questions answered. All CSVs exported to outputs/
